# Chapter 3 – LLM Embeddings

This notebook implements Sections 3.5.1 through 3.5.4:
- Generating embeddings with the Instructor model
- Comparing BERT vs LLM embeddings on sparse metadata
- Steering embeddings with different instructions
- Performance comparison

In [ ]:
%pip install sentence-transformers
%pip install python-dotenv

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

from recsys.data.loaders import (
    load_movielens,
    load_movielens_links,
    load_tmdb_movie_descriptions,
)
from recsys.utils.colab import get_data_path

## 1. Load data and prepare content text

Same data preparation as the Sentence Transformer notebook.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
tmdb_api_key = os.getenv("TMDB_API_KEY")

DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
links = load_movielens_links("ml-25m", data_dir=DATA_PATH)

descriptions = load_tmdb_movie_descriptions(
    links=links,
    api_key=tmdb_api_key,
    data_dir=DATA_PATH,
    cache_filename="movielens_descriptions.csv",
    force_refresh=False,
)

desc_df = pd.DataFrame.from_dict(descriptions, orient='index')
desc_df.reset_index(names='movieId', inplace=True)
desc_df['movieId'] = desc_df['movieId'].astype(str)

movies = movies.merge(desc_df[['movieId', 'overview']], on='movieId', how='left')
movies['overview'] = movies['overview'].fillna('')
movies['content'] = movies['title'] + ' ' + movies['genres'] + ' ' + movies['overview']

## 2. Load both models

In [ ]:
# BERT baseline (from Section 3.4)
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"BERT model dimensions: {bert_model.get_sentence_embedding_dimension()}")

# Instructor model (Section 3.5)
instructor_model = SentenceTransformer('hkunlp/instructor-xl')  #A
print(f"Instructor model dimensions: {instructor_model.get_sentence_embedding_dimension()}")

#A 1.3B parameter model, runs locally

## 3. Generate embeddings for all movies (Listing 3.16)

In [ ]:
# BERT embeddings (for comparison)
bert_embeddings = bert_model.encode(
    movies['content'].tolist(),
    show_progress_bar=True,
    batch_size=32
)
print(f"BERT embeddings: {bert_embeddings.shape}")

In [ ]:
# Listing 3.16: Generating Instructor embeddings for all movies
llm_embeddings = instructor_model.encode(
    [[
        "Represent the movie for retrieval:",  #A
        content
    ] for content in movies['content'].tolist()],
    show_progress_bar=True,
    batch_size=32
)  #B

#A Instruction guides what aspects the embedding should emphasize
#B Generate embeddings for all movies

print(f"Instructor embeddings: {llm_embeddings.shape}")

## 4. Comparing BERT vs Instructor on sparse metadata (Table 3.3)

Using only the query text "The Prestige" with no plot description,
we test which model handles sparse metadata better.

In [ ]:
def find_similar_from_query(query_embedding, embeddings, movies_df, k=5):
    """Find movies similar to a query embedding."""
    similarities = cosine_similarity(
        query_embedding.reshape(1, -1), embeddings
    )[0]
    top_indices = np.argsort(similarities)[-(k):][::-1]
    top_scores = similarities[top_indices]

    print()
    for i, (idx, score) in enumerate(zip(top_indices, top_scores), 1):
        movie = movies_df.iloc[idx]
        print(f"{i}. {movie['title']} sim: {score:.3f}, Genres: {movie['genres']}")

    return top_indices, top_scores

In [ ]:
sparse_query = "The Prestige"

# BERT
bert_query = bert_model.encode([sparse_query])[0]
print(f"BERT results for '{sparse_query}':")
find_similar_from_query(bert_query, bert_embeddings, movies, k=5)

# Instructor
instructor_query = instructor_model.encode([[
    "Represent the movie for retrieval:",
    sparse_query
]])[0]
print(f"\nInstructor results for '{sparse_query}':")
find_similar_from_query(instructor_query, llm_embeddings, movies, k=5)

## 5. Steering embeddings with instructions (Listing 3.17, Table 3.4)

The same seed movie (The Prestige) with different instructions
produces different recommendations.

In [ ]:
# Get The Prestige's full content text
prestige_id = movies[movies['title'] == "Prestige, The (2006)"]['movieId'].values[0]
prestige_desc = movies[movies['movieId'] == prestige_id]['content'].values[0]
print(f"Content text:\n{prestige_desc[:200]}...")

In [ ]:
# Listing 3.17: Steering embeddings with different instructions
mystery_instruction = "Represent the movie focusing on mystery and plot twists:"  #A
mystery_embedding = instructor_model.encode(
    [[mystery_instruction, prestige_desc]]
)[0]

rivalry_instruction = "Represent the movie focusing on rivalry and obsession:"  #B
rivalry_embedding = instructor_model.encode(
    [[rivalry_instruction, prestige_desc]]
)[0]

#A Create embedding focusing on mystery
#B Focus on rivalry and obsession

print('"Mystery and plot twists" results:')
find_similar_from_query(mystery_embedding, llm_embeddings, movies, k=5)

print('\n"Rivalry and obsession" results:')
find_similar_from_query(rivalry_embedding, llm_embeddings, movies, k=5)

## 6. Performance comparison (Section 3.5.4)

BERT is significantly faster than the Instructor model.

In [ ]:
sample_texts = movies['content'].head(100).tolist()

# BERT speed
start = time.time()
bert_model.encode(sample_texts, show_progress_bar=False)
bert_time = time.time() - start

# Instructor speed
start = time.time()
instructor_model.encode(
    [["Represent the movie for retrieval:", t] for t in sample_texts],
    show_progress_bar=False
)
instructor_time = time.time() - start

print(f"Embedding 100 movies:")
print(f"  BERT (MiniLM):   {bert_time:.1f}s ({bert_time/100*1000:.0f}ms per movie)")
print(f"  Instructor (XL): {instructor_time:.1f}s ({instructor_time/100*1000:.0f}ms per movie)")
print(f"  Instructor is {instructor_time/bert_time:.1f}x slower")

## 7. Cache LLM embeddings

In [ ]:
from pathlib import Path

cache_path = Path(DATA_PATH) / "movie_embeddings_instructor.npz"

np.savez_compressed(
    cache_path,
    embeddings=llm_embeddings,
    movie_ids=movies['movieId'].values
)
print(f"Saved Instructor embeddings to {cache_path}")
print(f"File size: {cache_path.stat().st_size / (1024**2):.1f} MB")